<a href="https://colab.research.google.com/github/zpempera/Social-Media-Analysis/blob/main/Social_media_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Libraries

In [ ]:
!pip install -q praw networkx python-louvain pyvis transformers spacy wordcloud
!python -m spacy download en_core_web_sm -q
!pip install feedparser

In [ ]:
import os
for folder in ["data/raw", "data/processed", "results"]:
    os.makedirs(folder, exist_ok=True)

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import json
import time
import feedparser
import urllib.parse

**Import data**

In [ ]:
SUBREDDITS = [
    "singularity", "OpenAI", "ChatGPT", "technology",
    "Futurology", "jobs", "antiwork", "ArtistHate"
]

QUERIES = [
    '"artificial intelligence" "future of work"',
    'ChatGPT education creativity',
    '"generative AI" threat opportunity',
    'AI ethics regulation misinformation',
    'automation "job loss" unemployment'
]

def search_reddit_rss(subreddit, query):
    q = urllib.parse.quote(query)
    url = f"https://www.reddit.com/r/{subreddit}/search.rss?q={q}&restrict_sr=1&sort=top&t=year"

    feed = feedparser.parse(url)
    rows = []

    for entry in feed.entries:
        rows.append({
            "subreddit": subreddit,
            "query": query,
            "title": entry.get("title"),
            "url": entry.get("link"),
            "published": entry.get("published"),
            "summary": entry.get("summary")
        })

    return rows

all_posts = []

for subreddit in SUBREDDITS:
    for query in QUERIES:
        print(f"Searching r/{subreddit}: {query}")
        rows = search_reddit_rss(subreddit, query)
        all_posts.extend(rows)
        time.sleep(6)

df = pd.DataFrame(all_posts)
df = df.drop_duplicates(subset=["url"])

print(df.shape)
display(df.head())

**Comments download**

In [ ]:
# =========================
# COMMENTS DOWNLOAD (RSS)
# =========================

from bs4 import BeautifulSoup

comments_data = []

# HTML cleaning
def clean_html(text):
    if pd.isna(text):
        return ""
    return BeautifulSoup(text, "html.parser").get_text(" ")

# Download comments for each post
for i, post_url in enumerate(df["url"]):

    try:
        comments_rss = post_url.rstrip("/") + ".rss"

        feed = feedparser.parse(comments_rss)

        for entry in feed.entries[:30]:   # max 30 comments per post

            comments_data.append({
                "post_url": post_url,
                "comment_author": entry.get("author"),
                "comment_text": clean_html(entry.get("summary")),
                "comment_published": entry.get("published"),
                "comment_link": entry.get("link")
            })

        print(f"Processed {i+1}/{len(df)} posts")

    except Exception as e:
        print(f"Error for {post_url}: {e}")

    # VERY IMPORTANT -> avoid rate limiting
    time.sleep(4)

# Create dataframe
comments_df = pd.DataFrame(comments_data)

# Remove duplicates
comments_df = comments_df.drop_duplicates(
    subset=["comment_text"]
)

print(comments_df.shape)

display(comments_df.head())

**Raw data into csv file**

In [ ]:
posts_df.to_csv("data/raw/posts.csv", index=False)
comments_df.to_csv("data/raw/comments.csv", index=False)
print(f"Posts: {len(posts_df)} | Comments: {len(comments_df)}")

# Od tego momentu możecie wczytywać dane z dysku zamiast z API:
# posts_df = pd.read_csv("data/raw/posts.csv")
# comments_df = pd.read_csv("data/raw/comments.csv")